In [ ]:
import pymysql
import pandas as pd
import re
import os

# ── 1. 전처리 (개행 제거) ──────────────────────
df = pd.read_csv("diningcode_data_crawling_gyeonggi_20260322_1244.csv")

def clean_text(v):
    if pd.isna(v): return ""
    return re.sub(r"\r\n|\r|\n", " ", str(v)).strip()

df["user_name"]  = df["user_name"].map(clean_text)
df["user_query"] = df["user_query"].map(clean_text)
df["menu"]       = df["menu"].map(clean_text)
df["item_name"]  = df["item_name"].map(clean_text)

clean_csv = "C:/Users/Public/diningcode_clean.csv"   # ← 슬래시 사용
df.to_csv(clean_csv, index=False, encoding="utf-8-sig")


# ── 2. DB 접속 ────────────────────────────────
conn = pymysql.connect(
    host="127.0.0.1", port=3306,
    user="root", password="*********",
    db="diningcode", charset="utf8mb4",
    local_infile=True
)
cur = conn.cursor()

# ── 3. 테이블 생성 ────────────────────────────
cur.execute("""
CREATE TABLE IF NOT EXISTS diningcode_gyeonggi (
    id                  INT,
    item_name           VARCHAR(100),
    item_area           VARCHAR(50),
    item_avg_rating     FLOAT,
    item_spec_area      VARCHAR(200),
    user_name           VARCHAR(100),
    user_tot_avg_rating FLOAT,
    user_tot_rating_num INT,
    user_tot_follow_num INT,
    user_rating         FLOAT,
    user_query          TEXT,
    taste               TINYINT  COMMENT '0부족 1보통 2좋음',
    price               TINYINT  COMMENT '0불만 1보통 2만족',
    service             TINYINT  COMMENT '0나쁨 1보통 2좋음',
    menu                TEXT,
    reviewed_at         DATE
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")

# ── 4. LOAD DATA LOCAL INFILE ─────────────────
cur.execute(f"""
LOAD DATA LOCAL INFILE 'C:/Users/Public/diningcode_clean.csv'
INTO TABLE diningcode_gyeonggi
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 LINES
(
    id, item_name, item_area, item_avg_rating, item_spec_area,
    user_name, user_tot_avg_rating, user_tot_rating_num, user_tot_follow_num,
    @raw_rating, user_query,
    @raw_taste, @raw_price, @raw_service,
    menu, @raw_date
)
SET
    user_rating = NULLIF(REGEXP_REPLACE(@raw_rating, '[^0-9.]', ''), ''),
    taste       = CASE @raw_taste
                    WHEN '맛: 좋음' THEN 2 WHEN '맛: 보통' THEN 1 WHEN '맛: 부족' THEN 0
                    ELSE NULL END,
    price       = CASE @raw_price
                    WHEN '가격: 만족' THEN 2 WHEN '가격: 보통' THEN 1 WHEN '가격: 불만' THEN 0
                    ELSE NULL END,
    service     = CASE @raw_service
                    WHEN '응대: 좋음' THEN 2 WHEN '응대: 보통' THEN 1 WHEN '응대: 나쁨' THEN 0
                    ELSE NULL END,
    reviewed_at = STR_TO_DATE(
                    REGEXP_REPLACE(@raw_date, '[년월 ]', '-'),
                    '%Y-%m-%d일'
                  );
""")

conn.commit()

# ── 5. 확인 ──────────────────────────────────
cur.execute("SELECT COUNT(*) FROM diningcode_gyeonggi")
print(f"✅ 삽입 완료: {cur.fetchone()[0]:,}행")

conn.close()
os.remove(clean_csv)

✅ 삽입 완료: 6,670행
